# Project 3 — Customer Segmentation (Unsupervised Learning)
### DecodeLabs Data Science Industrial Training Kit · Batch 2026

**Goal:** discover hidden customer segments in unlabeled retail order data using
distance-based algorithms: PCA to compress 20+ engineered features into 2-3
dimensions, K-Means clustering with the optimal K mathematically proven via the
Elbow Method + Silhouette Score, and a translation of the resulting clusters into
actionable business personas.

**Input:** this project reuses Project 2's order-level output
(`data/raw/project2_orders_with_fraud_features.csv`).

## ⚠️ A data-shape caveat, stated up front

This dataset has **1,200 orders across ~1,189 unique customers** — almost exactly
one order per customer. Real customer segmentation usually leans heavily on
repeat-purchase history (frequency, recency, loyalty). Here, that history barely
exists: only ~0.9% of customers placed more than one order.

Rather than ignore this, the aggregation step still builds those RFM-style columns
(`TotalOrders`, `TenureDays`, `IsRepeatCustomer`, `StdOrderValue`), but they are
**deliberately excluded from the clustering feature set** — a first attempt at this
pipeline showed that including them causes StandardScaler to blow their near-constant
values into huge z-scores for the rare 11 repeat customers, which then trivially
dominates PCA and produces a meaningless "repeat vs. everyone else" split instead of a
genuine segmentation. See `src/pipeline.py`'s `CLUSTERING_EXCLUDE_COLUMNS` for the
full reasoning, and the "trivial split" section below for a demonstration of exactly
what that looks like.

The clusters in this notebook should be read as **single-transaction spending
profiles** — what a customer bought, how much they spent, and how they paid — not
loyalty segments.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_orders
from src.customer_aggregation import build_customer_features, numeric_feature_columns
from src.dimensionality_reduction import reduce_dimensions
from src.clustering import evaluate_k_range, fit_final_clustering
from src.persona_builder import attach_cluster_labels, summarize_clusters, build_personas, personas_to_dataframe
from src.pipeline import CLUSTERING_EXCLUDE_COLUMNS, PROFILE_COLUMNS

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 40)

RAW_PATH = "../data/raw/project2_orders_with_fraud_features.csv" 

## 1. Load Orders & Aggregate to One Row Per Customer

In [2]:
df_orders = load_orders(RAW_PATH)
print(f"Orders: {df_orders.shape[0]}, Unique customers: {df_orders['CustomerID'].nunique()}")

customer_df = build_customer_features(df_orders)
print(f"Customer-level table: {customer_df.shape}")
customer_df.head()

Orders: 1200, Unique customers: 1189


Customer-level table: (1189, 38)


,CustomerID,TotalOrders,TotalSpend,AvgOrderValue,StdOrderValue,AvgQuantity,AvgUnitPrice,AvgItemsInCart,AvgCartFillRatio,AvgPricePerUnit,AvgItemValue,CouponUsageRate,WeekendOrderRate,HighValueOrderRate,FraudProxyRate,UniqueProducts,UniquePaymentMethods,AvgOrderMonth,AvgOrderDayOfWeek,RecencyDays,TenureDays,IsRepeatCustomer,TopProduct_Chair,TopProduct_Desk,TopProduct_Laptop,TopProduct_Other,TopProduct_Printer,TopProduct_Tablet,TopPayment_Cash,TopPayment_Credit Card,TopPayment_Debit Card,TopPayment_Gift Card,TopPayment_Online,TopReferral_Email,TopReferral_Facebook,TopReferral_Google,TopReferral_Instagram,TopReferral_Referral
0,C10002,1,1470.03,1470.03,0.0,3.0,490.01,4.0,0.750,490.01,367.508,1.0,0.0,0.0,1.0,1,1,10.0,0.0,624,0,0,False,False,False,False,True,False,True,False,False,False,False,False,True,False,False,False
1,C10054,1,153.48,153.48,0.0,2.0,76.74,7.0,0.286,76.74,21.926,1.0,0.0,0.0,1.0,1,1,10.0,0.0,260,0,0,False,False,True,False,False,False,True,False,False,False,False,False,False,False,True,False
2,C10126,1,1349.80,1349.80,0.0,2.0,674.90,4.0,0.500,674.90,337.450,1.0,1.0,0.0,0.0,1,1,6.0,6.0,744,0,0,False,False,False,True,False,False,False,False,False,False,True,False,True,False,False,False
3,C10154,1,1303.04,1303.04,0.0,4.0,325.76,8.0,0.500,325.76,162.880,1.0,0.0,0.0,0.0,1,1,7.0,0.0,337,0,0,False,True,False,False,False,False,False,False,False,True,False,True,False,False,False,False
4,C10211,1,635.90,635.90,0.0,5.0,127.18,10.0,0.500,127.18,63.590,1.0,0.0,0.0,1.0,1,1,5.0,3.0,418,0,0,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False


In [3]:
order_counts = customer_df["TotalOrders"].value_counts().sort_index()
print("Orders per customer distribution:")
print(order_counts)
print(f"\n{(order_counts.get(1,0) / len(customer_df) * 100):.1f}% of customers placed exactly 1 order.")

Orders per customer distribution:
TotalOrders
1    1178
2      11
Name: count, dtype: int64

99.1% of customers placed exactly 1 order.


## 2. Demonstration: Why Near-Constant Columns Must Be Excluded

Before finalizing the feature set, here's a direct demonstration of the failure
mode discovered during development: clustering on the FULL feature set (including
`TotalOrders`, `IsRepeatCustomer`, `TenureDays`, `StdOrderValue`) produces a trivial,
useless split.

In [4]:
all_feature_cols = numeric_feature_columns(customer_df)
X_all = customer_df[all_feature_cols]

pca_all = reduce_dimensions(X_all, max_components=3)
k_result_all = evaluate_k_range(pca_all.components, k_min=2, k_max=4)
_, labels_all, sil_all = fit_final_clustering(pca_all.components, k=2)

sizes = pd.Series(labels_all).value_counts().sort_index()
print("Cluster sizes WITH near-constant columns included:")
print(sizes)
print(f"Silhouette score: {sil_all:.4f}  <-- suspiciously high")
print(f"\nThis {sizes.min()}-customer 'cluster' is just the repeat-purchase customers,")
print("trivially isolated because their TotalOrders/TenureDays z-scores are enormous")
print("once scaled. This is not a useful business segmentation.")

Cluster sizes WITH near-constant columns included:
0    1178
1      11
Name: count, dtype: int64
Silhouette score: 0.8632  <-- suspiciously high

This 11-customer 'cluster' is just the repeat-purchase customers,
trivially isolated because their TotalOrders/TenureDays z-scores are enormous
once scaled. This is not a useful business segmentation.


**The fix:** exclude those 4 columns from clustering (they stay in
`customer_features.csv` for reference, just not used to group customers). Everything
from this point on uses the corrected feature set.

## 3. Feature Set Used for Clustering

In [5]:
feature_cols = [c for c in all_feature_cols if c not in CLUSTERING_EXCLUDE_COLUMNS]
print(f"Total engineered features: {len(all_feature_cols)}")
print(f"Excluded from clustering: {CLUSTERING_EXCLUDE_COLUMNS}")
print(f"Used for clustering: {len(feature_cols)}")
X = customer_df[feature_cols]
X.describe().T[["mean", "std", "min", "max"]]

Total engineered features: 37
Excluded from clustering: ['TotalOrders', 'IsRepeatCustomer', 'TenureDays', 'StdOrderValue']
Used for clustering: 33


,mean,std,min,max
TotalSpend,1063.390934,828.498360,11.390,5662.6875
AvgOrderValue,1055.201179,817.275488,11.390,3330.4075
AvgQuantity,2.947855,1.403689,1.000,5.0000
AvgUnitPrice,356.770063,196.816774,11.390,699.9300
AvgItemsInCart,5.489066,2.274300,1.000,10.0000
AvgCartFillRatio,0.579794,0.249308,0.167,1.0000
AvgPricePerUnit,356.709533,196.720383,11.390,699.9300
AvgItemValue,207.870775,153.824272,1.898,697.9300
CouponUsageRate,0.742641,0.435917,0.000,1.0000
WeekendOrderRate,0.298150,0.456487,0.000,1.0000


## 4. Phase 1-2: Scale & Compress (StandardScaler + PCA)

Every continuous feature is standardized to zero mean / unit variance first
(Euclidean distance otherwise lets large-magnitude features like `TotalSpend`
dominate), then PCA finds the orthogonal axes of maximum variance.

In [6]:
pca_result = reduce_dimensions(X, max_components=3)
print(f"Components needed for >=95% cumulative variance: {pca_result.n_components_95pct}")
print(f"Components kept for clustering (capped at 3 for visualization): {pca_result.n_components_kept}")
print(f"Cumulative variance explained by kept components: {pca_result.cumulative_variance[-1]:.3f}")

Components needed for >=95% cumulative variance: 22
Components kept for clustering (capped at 3 for visualization): 3
Cumulative variance explained by kept components: 0.258


In [7]:
fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(1, len(pca_result.cumulative_variance) + 1)
ax.plot(x, pca_result.cumulative_variance, marker="o")
ax.axhline(0.95, color="red", linestyle="--", label="95% threshold")
ax.set_xlabel("Number of Principal Components")
ax.set_ylabel("Cumulative Explained Variance")
ax.set_title("PCA: Cumulative Explained Variance (kept components only)")
ax.legend()
plt.tight_layout()
plt.show()

**Honest read:** with 33 candidate features and needing 22 components to reach
95% variance, this customer base doesn't have much redundancy to compress — each
feature contributes fairly independently. Capping at 3 components (per the brief's
"2 or 3 dimensions" requirement) keeps only ~26% of total variance. That's a real
trade-off: the 2D/3D visualization below is a genuine simplification, not a
near-complete picture, of the full feature space.

## 5. Phase 3: Proving the Optimal K (Elbow Method + Silhouette Score)

In [8]:
k_result = evaluate_k_range(pca_result.components, k_min=2, k_max=10)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(k_result.k_values, k_result.wcss, marker="o")
axes[0].axvline(k_result.elbow_k, color="red", linestyle="--", label=f"Elbow k={k_result.elbow_k}")
axes[0].set_xlabel("K"); axes[0].set_ylabel("WCSS (Inertia)"); axes[0].set_title("Elbow Method")
axes[0].legend()

axes[1].plot(k_result.k_values, k_result.silhouette_scores, marker="o", color="green")
axes[1].axvline(k_result.best_silhouette_k, color="red", linestyle="--", label=f"Best k={k_result.best_silhouette_k}")
axes[1].set_xlabel("K"); axes[1].set_ylabel("Silhouette Score"); axes[1].set_title("Silhouette Score")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Elbow Method suggests K = {k_result.elbow_k}")
print(f"Silhouette Score suggests K = {k_result.best_silhouette_k}")

Elbow Method suggests K = 5
Silhouette Score suggests K = 2


**The two gatekeepers disagree** (Elbow: K=5, Silhouette: K=2) — worth being
upfront about rather than picking whichever supports a predetermined answer. Looking
at the silhouette curve, scores across K=2 through K=10 are all fairly close (0.27 to
0.34, never much above "weak-to-moderate" cluster separation by the usual rule of
thumb). That's consistent with a customer base that sits on a **spending-level
continuum** rather than in a small number of sharply distinct segments. This project
uses the Silhouette Score as the deciding metric (it directly measures cluster
cohesion/separation, whereas the Elbow Method's "knee" can be ambiguous on real data),
giving a final choice of **K=2**.

## 6. Final Clustering & Visualization

In [9]:
kmeans_model, labels, final_silhouette = fit_final_clustering(pca_result.components, k_result.chosen_k)
print(f"Chosen K: {k_result.chosen_k}")
print(f"Final silhouette score: {final_silhouette:.4f}")

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(pca_result.components[:, 0], pca_result.components[:, 1],
                      c=labels, cmap="tab10", alpha=0.7, s=25)
ax.set_xlabel("Principal Component 1"); ax.set_ylabel("Principal Component 2")
ax.set_title("Customer Segments in PCA Space")
legend = ax.legend(*scatter.legend_elements(), title="Cluster")
ax.add_artist(legend)
plt.tight_layout()
plt.show()

Chosen K: 2
Final silhouette score: 0.3353


## 7. Phase 4: Translating Clusters into Business Personas

K-Means ran in abstract PCA space, which means nothing to a marketing team. This
step maps cluster assignments back onto the **original, human-readable** customer
features (not the PCA coordinates) to build the persona profile.

In [10]:
customer_with_clusters = attach_cluster_labels(customer_df, labels)
cluster_summary = summarize_clusters(customer_with_clusters, PROFILE_COLUMNS)
cluster_summary

,ClusterSize,PctOfCustomers,AvgOrderValue,TotalSpend,CouponUsageRate,WeekendOrderRate,HighValueOrderRate,FraudProxyRate,AvgQuantity,UniqueProducts
Cluster,,,,,,,,,,
0,441,37.1,1907.70,1921.72,0.74,0.30,0.13,0.42,3.66,1.00
1,748,62.9,552.59,557.34,0.75,0.29,0.00,0.41,2.53,1.01


In [11]:
personas = build_personas(cluster_summary)
personas_df = personas_to_dataframe(personas)

for p in personas:
    print(f"Cluster {p.cluster_id}: {p.name}  ({p.size} customers, {p.pct_of_customers}%)")
    print(f"  -> {p.recommended_action}")
    print()

Cluster 0: Full-Price Big Spenders  (441 customers, 37.1%)
  -> Prioritize for early access / premium loyalty perks — high spend without needing a discount to convert. Weekend-heavy ordering — time promotions for Friday/Saturday.

Cluster 1: Budget-Conscious Deal Hunters  (748 customers, 62.9%)
  -> Run frequent flash sales / coupon campaigns — this segment converts primarily on discounts.



In [12]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(personas_df["persona_name"], personas_df["size"], color="#4C72B0")
axes[0].set_ylabel("Number of customers"); axes[0].set_title("Persona Sizes")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(personas_df["persona_name"], personas_df["AvgOrderValue"], color="#DD8452")
axes[1].set_ylabel("Average Order Value ($)"); axes[1].set_title("Average Order Value by Persona")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 8. Conclusion

- 1,189 customers aggregated into a 33-feature behavioral profile table (well above
  the brief's "20+ columns" requirement).
- A trivial-split failure mode was found, diagnosed, and fixed: near-constant
  frequency columns (only 0.9% of customers are repeat buyers) were excluded from
  clustering after they were shown to dominate scaled distance and produce a
  meaningless 99%/1% split.
- PCA compressed the remaining 33 features to 3 components (~26% variance retained)
  for visualization and clustering.
- K was proven mathematically via both the Elbow Method (K=5) and Silhouette Score
  (K=2) — the two disagreed, and that disagreement is reported honestly rather than
  hidden, with the Silhouette Score used as the deciding metric.
- The final 2-cluster solution splits customers into **Full-Price Big Spenders**
  (37.1%) and **Budget-Conscious Deal Hunters** (62.9%), each with a concrete,
  actionable recommendation.

Outputs:
- `data/processed/customer_features.csv` — full customer-level feature table
- `data/processed/customer_segments.csv` — customer features + assigned cluster
- `reports/k_selection_diagnostics.csv`, `reports/clustering_summary.csv`,
  `reports/cluster_profile_summary.csv`, `reports/customer_personas.csv`
- `reports/figures/*.png` — PCA variance, elbow/silhouette, cluster scatter, persona matrix